# 25. 신규 관세평가 사례 확장 검색 실험

기존 177건에서 도출한 **144개 평가 사례·29개 제품군·Leave-One-Product-Family-Out 분할을 고정**합니다. 신규 MTI 227 사례 89건은 평가 데이터에 편입하지 않고, Fold별 검색 후보에만 추가합니다.

비교 목적은 신규 판정사례가 기존 평가 사례의 HS6 검색 순위와 분류 성능을 개선하는지 확인하는 것입니다. 기존 기준선과 확장 검색 결과를 같은 OOF 평가 사례에서 비교합니다.

- 기준 데이터: 기존 177건
- 고정 평가 데이터: 기존 데이터에서 재현한 144건
- 확장 검색 후보: 기존 평가 학습 후보 + 신규 89건
- Fold별 누수 방지: 평가 제품군과 동일한 제품군의 기존·신규 사례를 검색 후보에서 모두 제외
- 신규 사례는 HS6 후보 클래스가 기존 평가 대상 클래스에 속하는 경우에만 검색에 사용합니다.

주의: 신규 89건의 Product Type은 영문 품명·물품설명을 기준으로 거칠게 정규화한 검색용 그룹입니다. 최종 법적 분류를 새로 판정한 것이 아닙니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

In [ ]:
import re, json, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
BASE_FILE=quote('완제품_모델_기존177건_기준데이터.csv',safe='')
NEW_FILE=quote('CLIP_MTI227_신규89건_모델입력.csv',safe='')
KB_FILE='hs_knowledge_base.jsonl'
df=pd.read_csv(BASE+BASE_FILE,encoding='utf-8-sig').fillna('')
new_df=pd.read_csv(BASE+NEW_FILE,encoding='utf-8-sig').fillna('')
kb=pd.read_json(BASE+KB_FILE,lines=True).fillna('')
INPUT_COLUMNS=['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET='HS Code'
SUMMARY='Preprocessed Ruling Summary'
for data in [df,new_df]:
    for col in INPUT_COLUMNS+[TARGET,SUMMARY,'Product Type','Ruling Summary']: 
        if col not in data: data[col]=''
        data[col]=data[col].astype(str).str.strip()
df=df[df[TARGET].ne('')].copy()
df['duplicate_group']=pd.util.hash_pandas_object(df[INPUT_COLUMNS],index=False).astype(str)
class_counts=df[TARGET].value_counts()
eligible_labels=class_counts[class_counts>=4].index
eval_df=df[df[TARGET].isin(eligible_labels)].copy()
group_counts=eval_df.groupby(TARGET)['duplicate_group'].nunique()
eligible_labels=group_counts[group_counts>=4].index
eval_df=eval_df[eval_df[TARGET].isin(eligible_labels)].reset_index(drop=True)
for data in [eval_df,new_df]:
    data['hs_digits']=data[TARGET].map(lambda x:re.sub(r'\D','',str(x)))
    data['hs6']=data['hs_digits'].str[:6]
    data['hs4']=data['hs_digits'].str[:4]
    data['query_text']=data[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
    data['model_text']='[PRODUCT] '+data['Product Name']+' [INGREDIENTS_KR] '+data['Main Ingredients (KR)']+' [INGREDIENTS_EN] '+data['Main Ingredients (EN)']+' [FORM] '+data['Product Form and Packaging']+' [USE] '+data['Intended Use']
print('고정 평가 사례:',len(eval_df),'| HS6:',eval_df['hs6'].nunique(),'| 신규 검색 후보:',len(new_df))

## 1. 의미 기반 제품군 정의

원본 Product Type은 109종으로 세분되어 있고 단일 사례인 유형이 많습니다. 따라서 단순 Product Type 단위 분리 대신, 유사한 제품을 상위 제품군으로 묶습니다.

제품군 정의는 HS 코드가 아니라 제품의 형태·용도·제품 유형을 기준으로 합니다. 아래 규칙은 모든 평가 사례에 제품군을 부여하며, 매핑되지 않은 유형이 있으면 실험을 중단합니다.

In [ ]:
def assign_product_family(value):
    x=str(value).strip()
    if x in {'아이 메이크업 리무버','눈·입술 메이크업 제거'}: return 'eye_makeup_remover'
    if x in {'립 메이크업'}: return 'lip_makeup'
    if x in {'아이 메이크업'}: return 'eye_makeup'
    if x in {'화장품 함침 부직포'}: return 'impregnated_nonwoven_mask'
    if x in {'두발용 제품'}: return 'hair_care_styling'
    if x in {'영유아용 화장품'}: return 'baby_skincare'
    if x in {'방향·탈취 조제품'}: return 'indoor_air_freshener'
    if x in {'기초·기능성 화장품'}: return 'functional_skincare'
    if x in {'향수·화장수','네일용 제품','인체용 탈취제','목욕용 조제품','비누·세정용 제품','기타 화장용 제품','동물용 화장품'}: return 'basic_skincare'
    if '피내 주사용' in x or '필러' in x: return 'injectable_filler'
    if '향을 첨가한 종이' in x: return 'perfumed_paper'
    if '방향' in x: return 'vehicle_air_freshener' if '차량' in x else 'indoor_air_freshener'
    if '치과' in x or '불소 도포' in x or '산부식' in x: return 'dental_fluoride' if '불소' in x else 'dental_etchant'
    if '목욕' in x: return 'bath_foam' if '거품' in x else 'bath_oil_tablet'
    if '샴푸' in x: return 'shampoo_combo' if '겸용' in x else 'shampoo'
    if '염모' in x or '염색' in x or '컬러스프레이' in x: return 'hair_dye'
    if '헤어' in x or '두발' in x or '두피' in x: return 'hair_care_styling'
    if '립' in x or '입술' in x: return 'lip_balm' if '밤' in x else 'lip_exfoliant' if '스크럽' in x else 'lip_makeup'
    if '속눈썹' in x: return 'eyelash_care'
    if '마스카라' in x or '아이라이너' in x or '아이브로우' in x or '눈 화장' in x: return 'eye_makeup'
    if '볼연지' in x or '블러셔' in x: return 'blush'
    if '파우더' in x or '아이섀도' in x: return 'face_powder'
    if '부직포' in x or '물휴지' in x or '물티슈' in x or '마스크 팩' in x or '마스크팩' in x:
        return 'cosmetic_wipes' if '클렌징' in x or '물휴지' in x or '물티슈' in x else 'impregnated_nonwoven_mask'
    if '외음부' in x or '피어싱' in x or '이어홀' in x: return 'personal_cleansing'
    if '영유아' in x: return 'baby_skincare'
    if '클렌징' in x: return 'skincare_cleansing'
    if '선크림' in x or '자외선' in x: return 'suncare'
    if '팩' in x or '패치' in x or '마스크' in x: return 'skincare_mask_patch'
    if '기능성' in x: return 'functional_skincare'
    return 'basic_skincare'

eval_df['product_family']=eval_df['Product Type'].map(assign_product_family)
if eval_df['product_family'].isna().any(): raise ValueError('기존 평가 데이터 제품군 미분류')
labels=sorted(eval_df['hs6'].unique())
new_df=new_df[new_df['hs6'].isin(labels)].copy()
new_df['product_family']=new_df['Product Type'].map(assign_product_family)
candidate_df=pd.concat([eval_df,new_df],ignore_index=True).reset_index(drop=True)
candidate_df['candidate_source']= ['baseline']*len(eval_df)+['new_89']*(len(candidate_df)-len(eval_df))
family_audit=eval_df.groupby('product_family').agg(n=('hs6','size'),product_types=('Product Type','nunique'),hs6_codes=('hs6',lambda x:', '.join(sorted(x.unique())))).sort_values('n',ascending=False)
new_family_audit=new_df.groupby('product_family').agg(new_n=('hs6','size'),hs6_codes=('hs6',lambda x:', '.join(sorted(x.unique()))))
display(family_audit)
display(new_family_audit)
print('기존 평가:',len(eval_df),'| 검색 후보(신규 포함):',len(candidate_df),'| 신규 후보 중 평가 클래스 일치:',len(new_df))

## 2. 그룹 분리 가능성 검증

각 HS 6단위 코드가 최소 두 개 이상의 제품군에 존재해야 합니다. 그래야 한 제품군을 통째로 평가에서 제외하더라도 해당 HS 6단위 코드가 학습 데이터에 남아 있어 분류 문제로 평가할 수 있습니다.

이 조건을 만족하지 않는 코드는 이번 일반화 평가 대상에서 임의로 제외하지 않고, 데이터 부족으로 별도 보고합니다.

In [ ]:
label_family_counts=eval_df.groupby('hs6')['product_family'].nunique().sort_values()
display(label_family_counts.rename('제품군 수').to_frame())
unsupported=label_family_counts[label_family_counts<2]
if len(unsupported): raise ValueError(f'둘 이상의 제품군에 존재하지 않는 HS6 코드: {unsupported.to_dict()}')
print('제품군 수:',eval_df['product_family'].nunique(),'| 고정 평가 데이터의 HS6 제품군 커버리지 확인 완료')

## 3. 임베딩 및 검색 지식베이스 준비

E5 임베딩은 사전학습 모델을 사용합니다. 임베딩을 미리 계산하더라도 각 평가 Fold에서는 해당 제품군의 문서를 검색 대상에서 제외하므로, 평가 제품의 판정요지가 학습 검색에 유입되지 않습니다.

HS 해설서 문서는 HS 코드가 명시된 문서만 후보 코드 순위 산출에 사용합니다. 문서의 호·소호 코드와 예측 대상 HS 6단위는 숫자 접두어 관계로 연결합니다.

In [ ]:
kiwi=Kiwi()
STOPWORDS={'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
def remove_stopword_morphs(text):
    text=str(text); chars=list(text)
    for token in kiwi.tokenize(text):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i]=''
    return ''.join(chars)

for col in ['document_text','hs_code','rule_type','source_sheet']:
    if col not in kb.columns: kb[col]=''
    kb[col]=kb[col].fillna('').astype(str).str.strip()
kb['hs_digits']=kb['hs_code'].map(lambda x:re.sub(r'\D','',str(x)))
kb_code=kb[kb['document_text'].ne('') & kb['hs_digits'].ne('')].reset_index(drop=True)
kb_code['search_text']=(kb_code['document_text']+' '+kb_code['rule_type']+' '+kb_code['source_sheet']).str.strip()
kb_code['search_text_sw']=kb_code['search_text'].map(remove_stopword_morphs)
MODEL_NAME='intfloat/multilingual-e5-base'
sbert=SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return sbert.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_queries(texts):
    return sbert.encode(['query: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
query_emb=encode_queries(eval_df['query_text'].tolist())
product_emb=encode_docs(eval_df['query_text'].tolist())
ruling_text=(eval_df['query_text']+' '+eval_df[SUMMARY].map(remove_stopword_morphs)).str.strip()
product_ruling_emb=encode_docs(ruling_text.tolist())
candidate_product_emb=encode_docs(candidate_df['query_text'].tolist())
candidate_ruling_text=(candidate_df['query_text']+' '+candidate_df[SUMMARY].map(remove_stopword_morphs)).str.strip()
candidate_product_ruling_emb=encode_docs(candidate_ruling_text.tolist())
kb_emb=encode_docs(kb_code['search_text_sw'].tolist())
print('평가 임베딩:',product_emb.shape,'| 확장 검색 임베딩:',candidate_product_emb.shape,'| KB:',kb_emb.shape,'| device:',sbert.device)

## 4. HS 6단위 후보 순위 함수

KB 문서 유사도를 HS6 코드별로 집계하는 방식을 분리해 구현합니다. 기존 max 방식은 기준선으로 유지하고, top-3 mean 및 top-5 mean을 추가합니다.

In [ ]:
def rank_case_codes(scores,train_codes):
    tmp=pd.DataFrame({'hs6':np.asarray(train_codes).astype(str),'score':np.asarray(scores)})
    return tmp.groupby('hs6')['score'].max().sort_values(ascending=False)

def code_compatible(doc_code,target_code):
    a=re.sub(r'\D','',str(doc_code)); b=re.sub(r'\D','',str(target_code))
    return bool(a and b and (a.startswith(b) or b.startswith(a)))

def classify_kb_rule(rule_type):
    x=str(rule_type).strip()
    if x in {'제외','제외·예외'}: return 'negative'
    if '제외' in x and '포함' in x: return 'mixed'
    return 'positive'

kb_code['evidence_polarity']=kb_code['rule_type'].map(classify_kb_rule)
 
import json
CONDITION_PATTERNS={
'C33-001':[r'천연\s*올레오레진',r'natural\s+oleoresin'],
'C33-009':[r'의약용\s*조제품',r'습진\s*치료',r'외과수술',r'신체검사',r'의료기기.*커플링',r'surgical\s+lubricant',r'medical\s+lubricant'],
'IE-001':[r'천연\s*올레오레진',r'natural\s+oleoresin'],'IE-002':[r'휘발성.*식물성\s*물질',r'식물성\s*추출물.*제1302호',r'plant\s+extract.*1302'],
'IE-003':[r'동물성\s*착색제',r'식물성\s*착색제',r'animal\s+colorant',r'vegetable\s+colorant'],
'IE-005':[r'변성\s*정유',r'희석제',r'증량제',r'조성이.*현저히.*달라',r'diluted\s+essential\s+oil'],
'IE-006':[r'포마드',r'pomade'],'IE-007':[r'바닐라.*올레오레진',r'vanilla\s+oleoresin'],
'IE-008':[r'화학적으로\s*단일한',r'chemically\s+pure\s+compound',r'single\s+chemical\s+compound'],
'IE-009':[r'정유.*혼합물',r'정유.*혼합',r'essential\s+oil\s+mixture',r'essential\s+oil\s+blend'],
'IE-010':[r'검터펜틴',r'우드터펜틴',r'황산터펜틴',r'turpentine'],
'IE-011':[r'토일렛\s*비니거',r'toilet\s+vinegar'],
'IE-012':[r'애프터셰이브',r'aftershave',r'인체용\s*탈취제',r'body\s+deodorant'],
'IE-013':[r'습진\s*치료',r'특정\s*피부질환\s*치료',r'의약용\s*조제품',r'medicated\s+eczema',r'medicinal\s+preparation'],
'IE-014':[r'발냄새\s*방지',r'foot\s+deodorant'],'IE-015':[r'동물.*손톱',r'동물.*발톱',r'animal.*nail'],
'IE-016':[r'인조손톱',r'인조\s*네일',r'artificial\s+nails',r'false\s+nails'],
'IE-017':[r'두피\s*이외.*체모',r'체모용',r'제모용',r'body\s+hair',r'depilatory'],
'IE-018':[r'면도용\s*비누',r'shaving\s+soap'],
'IE-019':[r'합성\s*유기계면활성제',r'피부\s*세정용.*계면활성제',r'synthetic\s+organic\s+surfactant'],
'IE-020':[r'향기나는\s*양초',r'향초',r'scented\s+candle',r'fragranced\s+candle'],
'R3301-03':[r'변성\s*정유',r'희석제',r'증량제',r'조성이.*현저히.*달라']}
def condition_status(product,doc):
    kid=str(doc.get('knowledge_id',''))
    strong=' '.join(str(product.get(c,'')) for c in ['Product Name','Intended Use','Product Type'])
    weak=' '.join(str(product.get(c,'')) for c in ['Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging'])
    if kid=='C33-009' and re.search(r'석유\s*젤리|petroleum\s+jelly',strong+' '+weak,re.I):
        full=strong+' '+weak
        retail=bool(re.search(r'소매\s*포장|retail\s*pack|retail\s*packaged',full,re.I))
        skin=bool(re.search(r'피부\s*보호|skin\s*protect',strong,re.I))
        nonretail=bool(re.search(r'벌크|산업용|비소매|bulk|industrial',full,re.I))
        if retail and skin: return 'exception_applies'
        if nonretail or (retail and str(product.get('Intended Use','')).strip() and not skin): return 'strong_match'
        return 'unknown'
    if kid=='C33-009':
        substrate=bool(re.search(r'종이|워딩|펠트|부직포|paper|wadding|felt|nonwoven',strong+' '+weak,re.I))
        soap=bool(re.search(r'비누|세제|soap|detergent',strong+' '+weak,re.I))
        if substrate and soap: return 'strong_match' if re.search(r'종이|워딩|펠트|부직포|paper|wadding|felt|nonwoven',strong,re.I) else 'weak_match'
    if kid=='IE-019':
        use_hit=bool(re.search(r'피부\s*세정|클렌징|세정용|skin\s*cleanser|cleansing',strong,re.I))
        ingredient_hit=bool(re.search(r'sodium laureth sulfate|sodium lauryl sulfate|sodium coco.?sulfate|sodium c14.?16 olefin sulfonate|cocamidopropyl betaine|합성\s*유기계면활성제',weak,re.I))
        form_hit=bool(re.search(r'액상|크림|liquid|cream',str(product.get('Product Form and Packaging','')),re.I))
        if use_hit and ingredient_hit and form_hit: return 'strong_match'
        if use_hit and ingredient_hit: return 'weak_match'
        return 'unknown'
    pats=CONDITION_PATTERNS.get(kid,[])
    if not pats: return 'unknown'
    if any(re.search(p,strong,re.I) for p in pats): return 'strong_match'
    if any(re.search(p,weak,re.I) for p in pats): return 'weak_match'
    return 'unknown'
def kb_doc_compatible(doc,label):
    code=re.sub(r'\D','',str(doc.get('hs_code','')))
    if code and code_compatible(code,label): return True
    return str(doc.get('knowledge_id','')) in {'C33-001','C33-009'} and str(label).startswith('33')
def rank_kb_codes_condition_aware(product,scores,gamma=0.5,weak_weight=0.5,strong_only=False,top_k=5):
    vals={}
    score_arr=np.asarray(scores)
    docs=kb_code.to_dict('records')
    statuses=[condition_status(product,d) if d.get('evidence_polarity')=='negative' else 'not_negative' for d in docs]
    for label in labels:
        compatible=np.asarray([kb_doc_compatible(d,label) for d in docs])
        pos=np.asarray([score_arr[j] for j,d in enumerate(docs) if compatible[j] and d.get('evidence_polarity')=='positive'])
        neg=[]
        for j,d in enumerate(docs):
            if not compatible[j] or d.get('evidence_polarity')!='negative': continue
            if statuses[j]=='strong_match': neg.append(float(score_arr[j]))
            elif statuses[j]=='weak_match' and not strong_only: neg.append(float(score_arr[j])*weak_weight)
        positive=float(np.sort(pos)[::-1][:top_k].mean()) if len(pos) else 0.0
        vals[label]=positive-gamma*(max(neg) if neg else 0.0)
    return pd.Series(vals,dtype=float).sort_values(ascending=False)

display(kb_code['evidence_polarity'].value_counts().rename('문서 수').to_frame())

def rank_kb_codes_aggregate(scores,method='max',top_k=3,alpha=0.5):
    values={}
    doc_codes=kb_code['hs_digits'].to_numpy()
    for label in labels:
        matched=np.asarray([float(scores[j]) for j,doc_code in enumerate(doc_codes) if code_compatible(doc_code,label)])
        if len(matched):
            matched=np.sort(matched)[::-1]
            if method=='max': values[label]=matched[0]
            elif method=='topk_mean': values[label]=matched[:top_k].mean()
            elif method=='max_mean_blend':
                values[label]=alpha*matched[0]+(1-alpha)*matched[:top_k].mean()
            elif method=='rank_weighted_mean':
                selected=matched[:top_k]
                weights=1/np.log2(np.arange(2,len(selected)+2))
                values[label]=np.average(selected,weights=weights)
            else: raise ValueError(f'지원하지 않는 집계 방식: {method}')
    return pd.Series(values,dtype=float).sort_values(ascending=False)

def rank_kb_codes_polarity(scores,gamma=0.0,top_k=5):
    values={}
    doc_codes=kb_code['hs_digits'].to_numpy()
    polarities=kb_code['evidence_polarity'].to_numpy()
    for label in labels:
        compatible=np.asarray([code_compatible(code,label) for code in doc_codes])
        positive_scores=np.asarray(scores)[compatible & (polarities=='positive')]
        negative_scores=np.asarray(scores)[compatible & (polarities=='negative')]
        mixed_count=int(np.sum(compatible & (polarities=='mixed')))
        if len(positive_scores):
            positive=np.sort(positive_scores)[::-1][:top_k].mean()
        else:
            positive=0.0
        negative=float(np.max(negative_scores)) if len(negative_scores) else 0.0
        values[label]=positive-gamma*negative
    return pd.Series(values,dtype=float).sort_values(ascending=False)

def weighted_rrf_rank(case_rank,kb_rank,k=60,kb_weight=1.0):
    case_pos={str(c):i+1 for i,c in enumerate(case_rank.index)}
    kb_pos={str(c):i+1 for i,c in enumerate(kb_rank.index)}
    values={c:1/(k+case_pos.get(c,100000))+kb_weight/(k+kb_pos.get(c,100000)) for c in labels}
    return pd.Series(values).sort_values(ascending=False)

def rrf_rank(case_rank,kb_rank,k=60):
    return weighted_rrf_rank(case_rank,kb_rank,k=k,kb_weight=1.0)

def top_codes(rank,n=3): return [str(x) for x in rank.index[:n]]

def make_vectorizer():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))
    ])

## 5. Leave-One-Product-Family-Out 평가

한 제품군 전체를 평가 데이터로 두고 나머지 제품군으로 학습합니다. 따라서 같은 제품군에 속한 제품이 학습과 평가에 동시에 들어가지 않습니다.

모든 HS 6단위 코드가 둘 이상의 제품군에 나타나는지 앞 단계에서 확인했으므로, 각 반복에서 평가 대상 코드가 학습 데이터에 남아 있는지도 추가 검증합니다.

## 26. 330499·330790 혼동 집중 개선 실험

25번 실험의 144개 평가 사례·29개 제품군·Leave-One-Product-Family-Out 분할을 그대로 유지합니다. 신규 89건도 검색 후보로만 사용합니다.

25번 결과에서 330499→330790 오분류가 반복되었고, 특히 injectable_filler 제품군에서 집중되었습니다. 이 실험은 KB의 Top-5 Mean 근거가 해당 혼동을 악화시키는지 확인하기 위해 KB 가중치를 전역적으로 조정하고, injectable_filler 제품군에서만 별도로 조정합니다.

- 전역 KB 가중치: 0.00, 0.10, 0.25, 0.40, 0.50, 0.75, 1.00
- 필러 제품군 적응형 가중치: injectable_filler에만 0.00~0.75 적용, 나머지는 0.75 고정
- 비교 기준: 신규 사례 확장 + 판정요지 검색( KB 가중치 0 ) 및 기존 25번 Top5Mean w75
- 목적: 성능을 보고 최적 가중치를 선택하는 탐색 실험이며, 독립적인 최종 성능 추정으로 해석하지 않습니다.

주의: injectable_filler 제품군별 가중치는 제품군을 사전에 식별할 수 있다는 가정에 기반합니다. 제품군 매핑이 잘못되면 효과도 달라집니다.

In [ ]:
logo=LeaveOneGroupOut()
splits=list(logo.split(eval_df,eval_df['hs6'],groups=eval_df['product_family']))
prediction_rows=[]
fold_rows=[]
for fold,(tr,te) in enumerate(splits,1):
    train,test=eval_df.iloc[tr].copy(),eval_df.iloc[te].copy()
    train_labels=set(train['hs6'])
    if not set(test['hs6']).issubset(train_labels): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS 6단위 코드')
    fold_family=test['product_family'].iloc[0]
    aug_base_mask=candidate_df.index.isin(tr) & candidate_df['candidate_source'].eq('baseline').to_numpy()
    aug_new_mask=candidate_df['candidate_source'].eq('new_89').to_numpy() & candidate_df['product_family'].ne(fold_family).to_numpy()
    aug_mask=aug_base_mask | aug_new_mask
    fold_rows.append({'fold':fold,'heldout_family':fold_family,'train_n':len(tr),'test_n':len(te),'hs6_train_coverage':len(train_labels),'augmented_candidates_excluded':int((candidate_df['candidate_source'].eq('new_89') & candidate_df['product_family'].eq(fold_family)).sum()),'augmented_candidates_used':int(aug_mask.sum())})
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(train['model_text'])
    x_test=vectorizer.transform(test['model_text'])
    svm=LinearSVC(C=1.0,class_weight='balanced').fit(x_train,train['hs6'])
    svm_scores=svm.decision_function(x_test)
    for local_i,global_i in enumerate(te):
        svm_rank=pd.Series(svm_scores[local_i],index=svm.classes_).sort_values(ascending=False)
        prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':'TFIDF_LinearSVC','predicted_hs6':str(svm_rank.index[0]),'top3_candidates':top_codes(svm_rank)})
    train_idx=np.asarray(tr)
    train_codes=train['hs6'].to_numpy()
    for global_i in te:
        q=query_emb[global_i]
        case_product=rank_case_codes(product_emb[train_idx]@q,train_codes)
        case_ruling=rank_case_codes(product_ruling_emb[train_idx]@q,train_codes)
        kb_scores=kb_emb@q
        kb_rank_max=rank_kb_codes_aggregate(kb_scores,method='max')
        kb_rank_top5=rank_kb_codes_aggregate(kb_scores,method='topk_mean',top_k=5)
        ranks={
            'SBERT_product_only':case_product,
            'SBERT_product_plus_ruling':case_ruling,
            'SBERT_KB_only':kb_rank_max,
            'SBERT_product_only_plus_KB':rrf_rank(case_product,kb_rank_max),
            'SBERT_product_plus_ruling_plus_KB':rrf_rank(case_ruling,kb_rank_max),
            'SBERT_product_plus_ruling_plus_KB_w25':weighted_rrf_rank(case_ruling,kb_rank_max,kb_weight=0.25),
            'SBERT_product_plus_ruling_plus_KB_w50':weighted_rrf_rank(case_ruling,kb_rank_max,kb_weight=0.50),
            'SBERT_product_plus_ruling_plus_KB_w75':weighted_rrf_rank(case_ruling,kb_rank_max,kb_weight=0.75),
            'SBERT_product_plus_ruling_plus_KB_top5mean_w50':weighted_rrf_rank(case_ruling,kb_rank_top5,kb_weight=0.50),
            'SBERT_product_plus_ruling_plus_KB_top5mean_w75':weighted_rrf_rank(case_ruling,kb_rank_top5,kb_weight=0.75)
        }
        kb_positive=rank_kb_codes_polarity(kb_scores,gamma=0.0,top_k=5)
        for weight in [0.50,0.75]:
            ranks[f'SBERT_product_plus_ruling_plus_KB_positive_top5_w{int(weight*100)}']=weighted_rrf_rank(case_ruling,kb_positive,kb_weight=weight)
        for gamma in [0.25,0.50,1.00]:
            kb_signed=rank_kb_codes_polarity(kb_scores,gamma=gamma,top_k=5)
            for weight in [0.50,0.75]:
                ranks[f'SBERT_product_plus_ruling_plus_KB_signed_g{int(gamma*100)}_w{int(weight*100)}']=weighted_rrf_rank(case_ruling,kb_signed,kb_weight=weight)
        for gamma in [0.50,1.00]:
            for weight in [0.50,0.75]:
                kb_soft=rank_kb_codes_condition_aware(eval_df.iloc[global_i],kb_scores,gamma=gamma,weak_weight=0.5,strong_only=False,top_k=5)
                kb_strict=rank_kb_codes_condition_aware(eval_df.iloc[global_i],kb_scores,gamma=gamma,weak_weight=0.0,strong_only=True,top_k=5)
                ranks[f'SBERT_product_plus_ruling_plus_KB_condition_soft_g{int(gamma*100)}_w{int(weight*100)}']=weighted_rrf_rank(case_ruling,kb_soft,kb_weight=weight)
                ranks[f'SBERT_product_plus_ruling_plus_KB_condition_strict_g{int(gamma*100)}_w{int(weight*100)}']=weighted_rrf_rank(case_ruling,kb_strict,kb_weight=weight)

        aug_codes=candidate_df.loc[aug_mask,'hs6'].to_numpy()
        aug_product=rank_case_codes(candidate_product_emb[aug_mask]@q,aug_codes)
        aug_ruling=rank_case_codes(candidate_product_ruling_emb[aug_mask]@q,aug_codes)
        ranks.update({
            'SBERT_augmented_product_only':aug_product,
            'SBERT_augmented_product_plus_ruling':aug_ruling,
            'SBERT_augmented_product_plus_ruling_plus_KB_w75':weighted_rrf_rank(aug_ruling,kb_rank_max,kb_weight=0.75),
            'SBERT_augmented_product_plus_ruling_plus_KB_top5mean_w75':weighted_rrf_rank(aug_ruling,kb_rank_top5,kb_weight=0.75)
        })

        for weight in [0.00,0.10,0.25,0.40,0.50,0.75,1.00]:
            ranks[f'SBERT_augmented_top5mean_global_w{int(weight*100):03d}']=weighted_rrf_rank(aug_ruling,kb_rank_top5,kb_weight=weight)
        for weight in [0.00,0.10,0.25,0.40,0.50,0.75]:
            family_weight=weight if fold_family=='injectable_filler' else 0.75
            ranks[f'SBERT_augmented_top5mean_fadaptive_filler_w{int(weight*100):03d}']=weighted_rrf_rank(aug_ruling,kb_rank_top5,kb_weight=family_weight)

        for model_name,rank in ranks.items():
            prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':model_name,'predicted_hs6':str(rank.index[0]) if len(rank) else '', 'top3_candidates':top_codes(rank)})

predictions=pd.DataFrame(prediction_rows)
fold_audit=pd.DataFrame(fold_rows)
display(fold_audit)
print('제품군 분리 수:',len(splits),'| OOF 예측:',len(predictions),'| 모델별 예측 수:',predictions.groupby('model').size().to_dict())

## 6. KB 커버리지 및 검색 근거 진단

330499와 330690 사례에서 해설서 지식베이스가 어떤 HS 코드 문서를 보유하고, 실제 검색 시 어떤 문서를 상위에 반환하는지 확인합니다. 검색 근거는 평가 사례별 Top-15 문서로 저장합니다.

In [ ]:
coverage_rows=[]
for label in labels:
    matched=kb_code[kb_code['hs_digits'].map(lambda x:code_compatible(x,label))]
    coverage_rows.append({'hs6':label,'compatible_kb_documents':len(matched),'positive_documents':int((matched['evidence_polarity']=='positive').sum()),'negative_documents':int((matched['evidence_polarity']=='negative').sum()),'mixed_documents':int((matched['evidence_polarity']=='mixed').sum())})
kb_coverage=pd.DataFrame(coverage_rows).sort_values('hs6').reset_index(drop=True)
display(kb_coverage)

evidence_rows=[]
for row_idx,row in eval_df.iterrows():
    scores=kb_emb@query_emb[row_idx]
    top_idx=np.argsort(scores)[::-1][:15]
    for rank,j in enumerate(top_idx,1):
        doc=kb_code.iloc[j]
        evidence_rows.append({'row_index':int(row_idx),'product_family':row['product_family'],'product_name':row['Product Name'],'true_hs6':row['hs6'],'query_rank':rank,'similarity':float(scores[j]),'document_hs_code':doc['hs_code'],'document_hs_digits':doc['hs_digits'],'rule_type':doc['rule_type'],'evidence_polarity':doc['evidence_polarity'],'condition_status':condition_status(row,doc),'source_sheet':doc['source_sheet'],'document_text':doc['document_text']})
kb_retrieval_evidence=pd.DataFrame(evidence_rows)
display(kb_retrieval_evidence.groupby(['true_hs6','evidence_polarity']).size().reset_index(name='retrieved_count'))
display(kb_retrieval_evidence.sort_values(['true_hs6','row_index','query_rank']).groupby(['true_hs6','row_index']).head(5))

## 7. HS 4단위·6단위 성능 평가

예측 목표는 HS 6단위입니다. HS 4단위는 필수 일치 조건으로 별도 계산하고, HS 10단위 일치 여부는 평가하지 않습니다.

Top-3는 상위 3개 HS 6단위 후보 중 정답이 포함되는 비율입니다.

In [ ]:
def metric_summary(part):
    yt=part['true_hs6'].astype(str)
    yp=part['predicted_hs6'].astype(str)
    top3=np.mean([str(t) in [str(x) for x in c] for t,c in zip(yt,part['top3_candidates'])])
    return {
        'n':len(part),
        'hs4_accuracy':np.mean(yt.str[:4].to_numpy()==yp.str[:4].to_numpy()),
        'hs6_accuracy':accuracy_score(yt,yp),
        'macro_f1_hs6':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),
        'weighted_f1_hs6':f1_score(yt,yp,average='weighted',zero_division=0),
        'top3_hs6_accuracy':top3
    }

summary=pd.DataFrame([{'model':name,**metric_summary(part)} for name,part in predictions.groupby('model')]).sort_values('macro_f1_hs6',ascending=False)
display(summary.round(4))
family_metrics=predictions.groupby(['model','heldout_family']).apply(lambda x:pd.Series(metric_summary(x)),include_groups=False).reset_index()
display(family_metrics.sort_values(['model','heldout_family']))

class_metric_rows=[]
for model_name,part in predictions.groupby('model'):
    yt=part['true_hs6'].astype(str)
    yp=part['predicted_hs6'].astype(str)
    p,r,f,s=precision_recall_fscore_support(yt,yp,labels=labels,zero_division=0)
    for code,precision,recall,f1_value,support in zip(labels,p,r,f,s):
        class_metric_rows.append({'model':model_name,'hs6':code,'n':int(support),'precision':precision,'recall':recall,'f1':f1_value})
class_metrics=pd.DataFrame(class_metric_rows)
display(class_metrics.sort_values(['model','hs6']))
display(pd.crosstab(predictions['heldout_family'],predictions['true_hs6']))

## 8. 클래스별 성능 진단

각 HS6 코드별 Precision, Recall, F1을 함께 저장해 소수 클래스와 과잉 예측 문제를 확인합니다.

In [ ]:

condition_audit_rows=[]
negative_docs=kb_code[kb_code['evidence_polarity'].eq('negative')]
for row_idx,row in eval_df.iterrows():
    for _,doc in negative_docs.iterrows():
        status=condition_status(row,doc)
        condition_audit_rows.append({'row_index':int(row_idx),'product_name':row['Product Name'],'true_hs6':row['hs6'],'knowledge_id':doc['knowledge_id'],'rule_hs_code':doc['hs_code'],'rule_type':doc['rule_type'],'condition_status':status,'document_text':doc['document_text']})
condition_audit=pd.DataFrame(condition_audit_rows)
display(condition_audit['condition_status'].value_counts(dropna=False).to_frame('count'))


In [ ]:
OUT=Path('/content/hs_product_family_expansion_25')
OUT.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUT/'25_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'25_model_comparison.csv',index=False,encoding='utf-8-sig')
family_metrics.to_csv(OUT/'25_family_metrics.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'25_class_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv(OUT/'25_fold_audit.csv',index=False,encoding='utf-8-sig')
family_audit.to_csv(OUT/'25_product_family_audit.csv',encoding='utf-8-sig')
new_family_audit.to_csv(OUT/'25_new_candidate_family_audit.csv',encoding='utf-8-sig')
kb_coverage.to_csv(OUT/'25_kb_coverage_by_polarity.csv',index=False,encoding='utf-8-sig')
kb_retrieval_evidence.to_csv(OUT/'25_kb_retrieval_evidence_all_144.csv',index=False,encoding='utf-8-sig')
condition_audit.to_csv(OUT/'25_condition_rule_audit.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile('/content/25_expanded_retrieval_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob('*.csv'): z.write(f,arcname=f.name)
print('저장 완료: /content/25_expanded_retrieval_results.zip')

- 기존 177건에서 도출한 144개 평가 사례·29개 제품군을 고정합니다.
- 신규 89건은 평가에 포함하지 않고 검색 후보로만 추가합니다.
- 신규 사례의 HS6가 기존 평가 클래스에 포함되는 경우에만 검색 후보로 사용합니다.
- Fold별로 평가 제품군에 속한 모든 신규·기존 후보를 제외합니다.
- 기존 모델과 확장 모델의 성능은 동일한 OOF 평가 사례를 기준으로 비교합니다.
- 신규 사례의 Product Type은 검색 누수 방지용 제품군 정규화이며, 법적 분류 판정이 아닙니다.
- 제품군 매핑은 결과에 영향을 줄 수 있으므로 신규 Product Type 매핑표를 함께 검토해야 합니다.
- 신규 사례는 판정요지 기반 검색에 활용하므로, 학습 데이터로 재학습하는 지도학습 실험과는 구분합니다.
- OOF 결과를 보고 집계 방식을 선택한 성능은 선택 편향을 포함할 수 있습니다.

## 추가 해석 가이드

1. 전체 Macro-F1뿐 아니라 330499와 330790의 Precision·Recall·F1을 함께 비교합니다.
2. 330499→330790과 330790→330499의 혼동 건수를 별도로 확인합니다.
3. 필러 제품군에서만 가중치를 낮춘 모델이 전체 지표를 개선하는지, 다른 제품군 성능을 훼손하는지 확인합니다.
4. 최적 가중치는 동일 OOF 결과에 맞춘 탐색값이므로, 최종 보고에는 탐색 결과로 명시하고 추가 데이터 검증을 권장합니다.